# 01 — Exploratory Data Analysis & Baseline RAG

This notebook covers:
1. Loading and exploring HaluEval and TruthfulQA datasets
2. Analyzing data distributions (faithful vs. hallucinated)
3. Setting up a baseline RAG pipeline
4. Evaluating baseline performance without hallucination detection

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

import pandas as pd
import numpy as np
from src.utils.data_loader import load_halueval, load_truthfulqa

## 1. Load Datasets

In [ ]:
# Load HaluEval dataset
halueval_data = load_halueval(split='test')
print(f'HaluEval samples: {len(halueval_data)}')

# Convert to DataFrame for analysis
df_halu = pd.DataFrame(halueval_data)
df_halu.head()

In [ ]:
# Load TruthfulQA dataset
truthfulqa_data = load_truthfulqa()
print(f'TruthfulQA samples: {len(truthfulqa_data)}')

df_tqa = pd.DataFrame(truthfulqa_data)
df_tqa.head()

## 2. Data Distribution Analysis

In [ ]:
# Label distribution
print('HaluEval label distribution:')
print(df_halu['label'].value_counts())
print(f'\nFaithful ratio: {(df_halu["label"] == 0).mean():.2%}')
print(f'Hallucinated ratio: {(df_halu["label"] == 1).mean():.2%}')

print('\nTruthfulQA label distribution:')
print(df_tqa['label'].value_counts())

In [ ]:
# Response length analysis
df_halu['response_length'] = df_halu['response'].str.len()
print('Response length stats by label:')
print(df_halu.groupby('label')['response_length'].describe())

## 3. Baseline RAG Pipeline

In [ ]:
from src.rag_pipeline.retriever import HybridRetriever
from src.rag_pipeline.generator import LLMGenerator

# Initialize components
retriever = HybridRetriever()
generator = LLMGenerator()

# Test with a sample
sample = halueval_data[0]
print(f"Query: {sample['query']}")
print(f"Context: {sample['context'][0][:200]}...")

In [ ]:
# Generate baseline response
response = generator.generate(
    query=sample['query'],
    context_chunks=sample['context']
)
print(f"Generated response: {response}")
print(f"Ground truth: {sample['response']}")

## 4. Baseline Evaluation (No Detection)

Run the baseline RAG pipeline on a subset and measure basic metrics.

In [ ]:
from src.evaluation.metrics import answer_relevance, latency_stats
import time

# Evaluate on small subset
n_eval = 20
eval_data = halueval_data[:n_eval]

latencies = []
for sample in eval_data:
    start = time.time()
    resp = generator.generate(query=sample['query'], context_chunks=sample['context'])
    latencies.append((time.time() - start) * 1000)

print('Baseline latency stats:')
print(latency_stats(latencies))